In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

In [ ]:
df = df.dropna(subset=["Poem", "Genre"])

df["Poem"] = df["Poem"].astype(str)

In [ ]:
X = df["Poem"].values
y = df["Genre"].values

print(len(X), len(y))
print(type(X[0]))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

print(len(X_train), len(y_train))


In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    token_pattern=r"\b[a-zA-Z]{2,}\b",
    min_df=1,
    max_df=0.9
)


In [ ]:
pipe = Pipeline([
    ("vectorizer", vectorizer),
    ("model", RandomForestClassifier(random_state=42))
])


In [ ]:
param_dist = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}

In [ ]:
random_search = RandomizedSearchCV(
    pipe,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="f1_macro",
    random_state=42,
    n_jobs=-1,
    error_score="raise"
)

random_search.fit(X_train, y_train)

In [ ]:
melhor = random_search.best_estimator_
print(random_search.best_params_)

In [ ]:
y_pred = melhor.predict(X_test)

print(classification_report(y_test, y_pred))

In [ ]:
vectorizer = melhor.named_steps["vectorizer"]
random_forest = melhor.named_steps["model"]

feature_names = vectorizer.get_feature_names_out()
feature_importances = random_forest.feature_importances_


In [ ]:
top_indices = np.argsort(feature_importances)[-10:]
top_words = feature_names[top_indices]
top_importances = feature_importances[top_indices]

plt.figure(figsize=(10, 6))
plt.barh(top_words, top_importances)
plt.xlabel("Importância")
plt.ylabel("Palavra")
plt.title("Top 10")
plt.show()

In [ ]:
y_pred = melhor.predict(X_test)

print(classification_report(y_test, y_pred))